# Week 7: Search Query Intent Classification

This notebook classifies real-estate queries from their wording into `browsing`, `researching`, or `high_intent_inquiry`. It uses no CRM, conversion, click, demographic, or buyer-behavior data.

## Labeling protocol

The dataset contains 240 unique, balanced, manually reviewed queries: 80 examples per class. It began as a seed dataset, and all labels in the current CSV are now marked `manual_verified`. The held-out score still reflects this curated distribution, so it is not a guarantee of performance on naturally written production queries.

In [1]:
from pathlib import Path
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'scripts').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the project directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.train_query_intent_classifier import load_dataset, train_and_evaluate
from scripts.query_intent_classifier import QueryIntentClassifier
from scripts.query_parser import QueryParser

dataset = load_dataset(ROOT / 'data/processed/week7_intent_queries.csv')
dataset.groupby(['intent', 'review_status']).size()

intent               review_status  
browsing             manual_verified    80
high_intent_inquiry  manual_verified    80
researching          manual_verified    80
dtype: int64

## TF-IDF + logistic regression

The model learns unigram and bigram wording patterns. Logistic regression produces a probability for every class, which is returned as a confidence score. The test split is stratified so each class is represented in both training and testing data.

In [2]:
classifier, metrics = train_and_evaluate(dataset)
print(f"Held-out accuracy: {metrics['accuracy']:.1%}")
print(f"Train/test rows: {metrics['train_size']}/{metrics['test_size']}")
print(metrics['classification_report'])

Held-out accuracy: 100.0%
Train/test rows: 192/48
                     precision    recall  f1-score   support

           browsing      1.000     1.000     1.000        16
high_intent_inquiry      1.000     1.000     1.000        16
        researching      1.000     1.000     1.000        16

           accuracy                          1.000        48
          macro avg      1.000     1.000     1.000        48
       weighted avg      1.000     1.000     1.000        48



## Integration with the Week 4 Query Parser

The parser extracts factual filters such as city, price, bedrooms, and amenities. The Week 7 classifier adds an intent and confidence score. These are complementary: the parser does not decide user intent, and intent classification does not replace factual filters.

In [3]:
examples = [
    'show me homes in Irvine',
    'which neighborhoods in San Diego have lower property taxes',
    'schedule a tour of 3 bedroom homes in Irvine under $1M this weekend',
]
parser = QueryParser()
for query in examples:
    result = classifier.classify_with_parser(query, parser)
    print(query)
    print({key: result[key] for key in ('intent', 'confidence', 'is_uncertain')})
    print(result['filters'])
    print()

show me homes in Irvine
{'intent': 'browsing', 'confidence': 0.7858043161891984, 'is_uncertain': False}
{'city': 'Irvine', 'price_min': None, 'price_max': None, 'beds_min': None, 'beds_max': None, 'baths_min': None, 'baths_max': None, 'amenities_include': [], 'amenities_exclude': [], 'terms_include': [], 'terms_exclude': [], 'matched_categories': []}

which neighborhoods in San Diego have lower property taxes
{'intent': 'researching', 'confidence': 0.8158995253241439, 'is_uncertain': False}
{'city': 'San Diego', 'price_min': None, 'price_max': None, 'beds_min': None, 'beds_max': None, 'baths_min': None, 'baths_max': None, 'amenities_include': [], 'amenities_exclude': [], 'terms_include': [], 'terms_exclude': [], 'matched_categories': []}

schedule a tour of 3 bedroom homes in Irvine under $1M this weekend
{'intent': 'high_intent_inquiry', 'confidence': 0.779218406855617, 'is_uncertain': False}
{'city': 'Irvine', 'price_min': None, 'price_max': 1000000, 'beds_min': 3, 'beds_max': 3, 'ba

## Limitations

- Intent is inferred only from query wording. It does **not** infer whether a person will buy, their financial ability, demographics, or actual conversion likelihood.
- All 240 labels were manually reviewed, but the examples remain curated and may contain template-like language. A high held-out score can overestimate performance on naturally written user queries.
- Future work should add naturally occurring, ambiguous queries from a separate source before making broader real-world claims.
- A low confidence score should be routed as uncertain rather than treated as a definitive label.